In [3]:
import pandas as pd
import numpy as np
import os

RAW_DIR = "../data/raw/"
PROCESSED_DIR = "../data/preprocessed/"

os.makedirs(PROCESSED_DIR, exist_ok=True)

edges_path  = RAW_DIR + "connections_princeton.csv"
nodes_path  = RAW_DIR + "names.csv"
labels_path = RAW_DIR + "labels.csv"


In [4]:
print("Loading raw files...")
edges_raw  = pd.read_csv(edges_path)
nodes_raw  = pd.read_csv(nodes_path)
labels_raw = pd.read_csv(labels_path)

print("Edges:", edges_raw.shape)
print("Nodes:", nodes_raw.shape)
print("Labels:", labels_raw.shape)

edges_raw.head()


Loading raw files...
Edges: (5342446, 5)
Nodes: (139255, 3)
Labels: (160045, 9)


,pre_root_id,post_root_id,neuropil,syn_count,nt_type
0,720575940625363947,720575940623224444,ME_L,12,GABA
1,720575940630432382,720575940618518557,ME_L,67,ACH
2,720575940627314521,720575940626337738,ME_L,10,GABA
3,720575940620280405,720575940620204726,ME_L,15,ACH
4,720575940636942447,720575940613789411,LA_L,1,GLUT


In [5]:
rename_map = {
    "pre_pt_root_id":  "pre",
    "post_pt_root_id": "post",
    "pre_root_id":     "pre",
    "post_root_id":    "post",
    "pre_segment_id":  "pre",
    "post_segment_id": "post",
    "syn_count":       "weight",
    "weight":          "weight",
    "w":               "weight",
}

edges = edges_raw.rename(columns={k: v for k, v in rename_map.items() if k in edges_raw.columns})


In [6]:
# Ensure required columns exist
required_cols = {"pre", "post"}
if not required_cols.issubset(edges.columns):
    raise ValueError("Missing required columns in edges: ", edges.columns)

# If weight missing → set weight = 1
if "weight" not in edges.columns:
    edges["weight"] = 1


In [7]:
edges = edges[["pre", "post", "weight"]].copy()

edges["pre"]    = edges["pre"].astype("int64")
edges["post"]   = edges["post"].astype("int64")
edges["weight"] = edges["weight"].astype(int)


In [8]:
edges = edges.groupby(["pre", "post"], as_index=False)["weight"].sum()
edges.head()


,pre,post,weight
0,720575940596125868,720575940605825666,6
1,720575940596125868,720575940608552405,6
2,720575940596125868,720575940609975854,5
3,720575940596125868,720575940613059993,9
4,720575940596125868,720575940613599129,6


In [9]:
node_rename = {}
if "root_id" in nodes_raw.columns:
    node_rename["root_id"] = "id"
if "pt_root_id" in nodes_raw.columns:
    node_rename["pt_root_id"] = "id"

nodes = nodes_raw.rename(columns=node_rename)

# If no ID → build from edges
if "id" not in nodes.columns:
    unique_ids = pd.Index(edges["pre"]).union(edges["post"])
    nodes = pd.DataFrame({"id": unique_ids})

nodes["id"] = nodes["id"].astype("int64")


In [10]:
if "name" not in nodes.columns:
    nodes["name"] = nodes["id"].astype(str)

if "group" not in nodes.columns:
    nodes["group"] = "Unknown"


In [11]:
label_per_root = (
    labels_raw.groupby(["root_id", "label"])
              .size()
              .reset_index(name="count")
              .sort_values(["root_id", "count"], ascending=[True, False])
              .drop_duplicates("root_id")
              .rename(columns={"label": "community_label"})[["root_id", "community_label"]]
)

nodes = nodes.merge(label_per_root, left_on="id", right_on="root_id", how="left")
nodes = nodes.drop(columns=["root_id"], errors="ignore")

nodes.head()


,id,name,group,community_label
0,720575940596125868,LO.LOP.561,LO.LOP,T5c; FBbt_00003739
1,720575940597856265,ME.2982,ME,"Tm16 (putative, Fischbach 1989)"
2,720575940597944841,ME.LO.7713,ME.LO,Tm7; Transmedullary neuron 7; FBbt_00003795
3,720575940598267657,ME.4244,ME,TmY15; Transmedullary Y neuron 15; FBbt_00048246
4,720575940599333574,ME.5182,ME,Tm1; Transmedullary neuron 1; FBbt_00003789


In [12]:
edges.to_csv(PROCESSED_DIR + "processed_edges.csv", index=False)
nodes.to_csv(PROCESSED_DIR + "processed_nodes.csv", index=False)

print("✔ Saved processed files to:", PROCESSED_DIR)


✔ Saved processed files to: ../data/preprocessed/
